<a href="https://colab.research.google.com/github/ArshAnan/llm-offload-controller/blob/main/notebooks/02_offload_env_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ArshAnan/llm-offload-controller.git
%cd llm-offload-controller

!git config user.email "arshanand2524@gmail.com"
!git config user.name "ArshAnan"

Mounted at /content/drive
Cloning into 'llm-offload-controller'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 55 (delta 23), reused 5 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 93.02 KiB | 2.66 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/llm-offload-controller


Add Data

In [3]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

# Load preprocessed training data from Drive
drive_path = "/content/drive/MyDrive/llm-offload-controller/data/"
train_df = pd.read_csv(drive_path + "train_processed.csv")

print(f"Loaded {len(train_df):,} rows")
print(train_df.columns.tolist())

Loaded 8,301,611 rows
['timestamp', 'model', 'req_tokens', 'resp_tokens', 'total_tokens', 'log_type', 'inter_arrival', 'hour_bin', 'arrival_rate', 'hour_of_day', 'is_gpt4', 'is_large_request', 'req_tokens_norm', 'inter_arrival_norm', 'arrival_rate_norm', 'rolling_req_tokens', 'rolling_inter_arrival']


In [4]:
# Define state columns
state_cols = [
    'req_tokens_norm',
    'rolling_req_tokens',
    'inter_arrival_norm',
    'rolling_inter_arrival',
    'arrival_rate_norm',
    'hour_of_day',
    'is_gpt4',
    'is_large_request'
]

print(train_df[state_cols].head(5))
print("\nAny nulls?", train_df[state_cols].isnull().sum().sum())

   req_tokens_norm  rolling_req_tokens  inter_arrival_norm  \
0         0.014809                 0.0            0.000000   
1         0.014809                 0.0            0.000000   
2         0.034395                 0.0            0.001077   
3         0.034395                 0.0            0.000000   
4         0.013057                 0.0            0.001966   

   rolling_inter_arrival  arrival_rate_norm  hour_of_day  is_gpt4  \
0                    0.0                0.0            0        0   
1                    0.0                0.0            0        0   
2                    0.0                0.0            0        0   
3                    0.0                0.0            0        0   
4                    0.0                0.0            0        1   

   is_large_request  
0                 0  
1                 0  
2                 1  
3                 1  
4                 0  

Any nulls? 0


Building Gym Environment

In [5]:
class OffloadEnv(gym.Env):
    def __init__(self, df, state_cols):
        super().__init__()

        self.df = df.reset_index(drop=True)
        self.state_cols = state_cols
        self.current_step = 0

        # Action space: 0 = local, 1 = edge, 2 = cloud
        self.action_space = spaces.Discrete(3)

        # State space: 8 normalized features
        self.observation_space = spaces.Box(
            low=0.0, high=1.0,
            shape=(len(state_cols),),
            dtype=np.float32
        )

    def reset(self, seed=None):
        self.current_step = 0
        obs = self.df[self.state_cols].iloc[0].values.astype(np.float32)
        return obs, {}

In [8]:
class OffloadEnv(gym.Env):
    def __init__(self, df, state_cols):
        super().__init__()

        self.df = df.reset_index(drop=True)
        self.state_cols = state_cols
        self.current_step = 0

        # Action space: 0 = local, 1 = edge, 2 = cloud
        self.action_space = spaces.Discrete(3)

        # State space: 8 normalized features
        self.observation_space = spaces.Box(
            low=0.0, high=1.0,
            shape=(len(state_cols),),
            dtype=np.float32
        )

    def reset(self, seed=None):
        self.current_step = 0
        obs = self.df[self.state_cols].iloc[0].values.astype(np.float32)
        return obs, {}

    def step(self, action):
        row = self.df.iloc[self.current_step]

        # Energy cost per tier
        energy_cost = {0: 1.0, 1: 2.5, 2: 6.0}

        # Latency cost per tier (seconds per token)
        latency_cost = {0: 0.001, 1: 0.0005, 2: 0.0002}

        # Queue pressure — how busy has local been recently?
        queue_pressure = row['rolling_req_tokens'] * row['is_large_request']

        # Local latency grows under load, edge/cloud stay consistent
        if action == 0:
            estimated_latency = latency_cost[0] * row['req_tokens'] * (1 + queue_pressure * 10)
        else:
            estimated_latency = latency_cost[action] * row['req_tokens']

        # P99 target from EDA
        p99_target = 2991

        # Reward — penalize tail latency and energy
        alpha = 1.0  # latency weight
        beta  = 0.1  # energy weight

        if estimated_latency > p99_target:
            reward = -alpha * (estimated_latency - p99_target) - beta * energy_cost[action]
        else:
            reward = -beta * energy_cost[action]

        # Move to next step
        self.current_step += 1
        done = self.current_step >= len(self.df) - 1

        next_obs = self.df[self.state_cols].iloc[self.current_step].values.astype(np.float32)

        return next_obs, reward, done, False, {}

In [9]:
# Test the environment with random actions
env = OffloadEnv(train_df, state_cols)

obs, _ = env.reset()
print(f"Initial observation: {obs}")
print(f"Observation shape: {obs.shape}")

# Take 5 random steps
for i in range(5):
    action = env.action_space.sample()  # random action: 0, 1, or 2
    next_obs, reward, done, _, _ = env.step(action)
    action_name = {0: 'local', 1: 'edge', 2: 'cloud'}[action]
    print(f"Step {i+1} | Action: {action_name} | Reward: {reward:.4f} | Done: {done}")

Initial observation: [0.01480892 0.         0.         0.         0.         0.
 0.         0.        ]
Observation shape: (8,)
Step 1 | Action: local | Reward: -0.1000 | Done: False
Step 2 | Action: cloud | Reward: -0.6000 | Done: False
Step 3 | Action: cloud | Reward: -0.6000 | Done: False
Step 4 | Action: edge | Reward: -0.2500 | Done: False
Step 5 | Action: edge | Reward: -0.2500 | Done: False
